### Accessing Data in LeMat-Rho AWS OpenData Repository
Data is stored in the following format:
/`immutable_ID`/`functional`_`calc_type`/`VASP_file`.json.gz

Here, the `immutable_ID` is similar to LeMat-Rho, in that it is the material ID respective of which database it came from, ie `mp-100` for Materials Project mp-100 material ID. The `functional` is the DFT functional used, for now it is only `r2scan` and the `calc_type` is the calculation type that was used to create the RAW vasp files (ie, `relax` for a relaxation calculation). `VASP_file` is the vasp input/output file name such as `CHGCAR` or `AECCAR0` for example. It has been parsed using Pymatgen, saved as a json dictionary-like object and gzipped. 

### Downloading Data from AWS

The following code below will download data from the AWS repository

In [10]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import json
import gzip
import io
from pymatgen.io.vasp import Chgcar

AWS_BUCKET_NAME = "lemat-rho"
FILE_PATH = 'mp-1/pbe_relax/CHGCAR.json.gz'


def stream_gz_file_from_aws_bucket(s3_key, processor_cls):
    s3 = boto3.client(
        "s3", region_name="us-west-2", config=Config(signature_version=UNSIGNED)
    )
    response = s3.get_object(Bucket=AWS_BUCKET_NAME, Key=s3_key)

    # Stream and decompress using GzipFile
    gzipped_body = gzip.GzipFile(fileobj=response["Body"])

    # If needed, wrap in BufferedReader to make it seekable
    buffered_reader = io.BufferedReader(gzipped_body)

    # Pass to processor
    processor = processor_cls(buffered_reader)
    return processor.process()


class ChgCarProcessor:
    def __init__(self, file_obj):
        self.file_obj = file_obj

    def process(self):
        self.chgcar = Chgcar.from_dict(
            json.loads(self.file_obj.read().decode("utf-8"))["data"]
        )
        return self.chgcar

chgcar = stream_gz_file_from_aws_bucket(
    s3_key=FILE_PATH, processor_cls=ChgCarProcessor)

### Using PyRho from Materials Project to convert/process charge density files
In the database on HuggingFace, we use [pyRho](https://materialsproject.github.io/pyrho/) to process the raw charge densities and upscale them to be similar in sizes

In [ ]:
from pyrho.charge_density import ChargeDensity
charge_density = ChargeDensity.from_pmg(chgcar)
upscaled_density = charge_density.pgrids['total'].lossy_smooth_compression([300,300,300])